# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook demonstrates probabilistic patient linkage using Splink across disparate EHR records (Fellegi-Sunter).

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.appName("Day05_Splink_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Create Mock Patient Data from Multiple EHR Systems
data = [
    {"unique_id": "EHR1_1", "first_name": "John", "last_name": "Doe", "dob": "1980-01-01", "city": "New York"},
    {"unique_id": "EHR2_1", "first_name": "Jon", "last_name": "Doe", "dob": "1980-01-01", "city": "New York"}, # Match for John Doe
    {"unique_id": "EHR1_2", "first_name": "Jane", "last_name": "Smith", "dob": "1992-11-02", "city": "Boston"},
    {"unique_id": "EHR3_1", "first_name": "J.", "last_name": "Smith", "dob": "1992-11-02", "city": "Boston"}, # Match for Jane Smith
    {"unique_id": "EHR1_3", "first_name": "Robert", "last_name": "Johnson", "dob": "1975-08-20", "city": "Chicago"}
]

df_patients = spark.createDataFrame(data)
df_patients.show()

In [ ]:
from splink.spark.linker import SparkLinker
import splink.spark.comparison_library as cl

# Define Linkage Model Settings
settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        "l.dob = r.dob",
        "l.city = r.city"
    ],
    "comparisons": [
        cl.exact_match("first_name", term_frequency_adjustments=True),
        cl.exact_match("last_name", term_frequency_adjustments=True),
        cl.exact_match("dob"),
        cl.exact_match("city")
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

# Initialize Splink Linker
linker = SparkLinker(df_patients, settings)

# Train the model using deterministic rules for estimation
linker.estimate_u_using_random_sampling(max_pairs=1e6)
linker.estimate_parameters_using_expectation_maximisation("l.dob = r.dob")
linker.estimate_parameters_using_expectation_maximisation("l.first_name = r.first_name")

# Generate Predictions
df_predictions = linker.predict(threshold_match_probability=0.5)

print("Probabilistic Linkage Results (Match Probability > 0.5):")
df_predictions.as_pandas_dataframe()[[
    'match_probability', 'unique_id_l', 'unique_id_r', 
    'first_name_l', 'first_name_r', 'last_name_l', 'last_name_r'
]]